In [1]:
import joblib

import gradio as gr 
import requests
import pandas as pd
from datetime import date, timedelta
import os
     


In [ ]:
model = joblib.load("/mnt/d/6thsem_dgp datasets/Model/crop_model.pkl")
scaler = joblib.load("/mnt/d/6thsem_dgp datasets/Model/scaler.pkl")
le = joblib.load("/mnt/d/6thsem_dgp datasets/Model/label_encoder.pkl")

In [21]:
FAO_RULES = {
    "rice": (20,35,1000,2000,5.0,7.5),
    "maize": (18,27,500,800,5.5,7.5),
    "jute": (24,35,1200,2500,5.0,7.5),
    "cotton": (21,35,500,1200,5.5,8.0),
    "coconut": (20,32,1500,2500,5.2,8.0),
    "papaya": (21,33,1000,2000,5.5,7.0),
    "orange": (15,30,900,1200,5.5,7.5),
    "apple": (5,20,600,1000,5.5,6.5),
    "muskmelon": (18,30,400,600,6.0,7.5),
    "watermelon": (20,35,400,600,6.0,7.5),
    "grapes": (15,30,500,900,6.0,7.5),
    "mango": (24,35,750,2500,5.5,7.5),
    "banana": (20,30,1000,2500,6.0,7.5),
    "pomegranate": (20,35,500,800,6.5,7.5),
    "lentil": (15,25,300,600,6.0,7.5),
    "blackgram": (25,35,600,1000,6.0,7.5),
    "mungbean": (25,35,600,1000,6.0,7.5),
    "mothbeans": (25,35,200,500,6.0,8.0),
    "pigeonpeas": (18,35,600,1000,5.5,7.5),
    "kidneybeans": (15,30,600,1200,5.5,7.5),
    "chickpea": (15,25,300,600,6.0,7.5),
    "coffee": (18,25,1200,2500,5.0,6.5),
}

def fao_validate(crop, temp, rain, ph):
    if crop not in FAO_RULES:
        return True
    tmin,tmax,rmin,rmax,pmin,pmax = FAO_RULES[crop]
    return tmin <= temp <= tmax and rmin <= rain <= rmax and pmin <= ph <= pmax


In [23]:
ICAR_RULES = {
    "gujarat": [
        "cotton","maize","chickpea",
        "pigeonpeas","mothbeans"
    ],
    "maharashtra": [
        "cotton","jowar","bajra",
        "wheat","mungbean"
    ],
    "punjab": [
        "rice","wheat","maize"
    ],
    "karnataka": [
        "coffee","rice","maize","banana"
    ],
    "kerala": [
        "rice","coconut","banana","coffee"
    ]
}

def icar_validate(crop, place):
    place = place.lower()
    for region, crops in ICAR_RULES.items():
        if region in place:
            return crop in crops
    return True


In [24]:
def get_lat_lon(place):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": place, "count": 1, "language": "en", "format": "json"}
    data = requests.get(url, params=params, timeout=10).json()
    if "results" not in data:
        return None, None
    return data["results"][0]["latitude"], data["results"][0]["longitude"]


In [25]:
def get_short_term_temp_humidity(lat, lon, days=14):
    end = date.today() - timedelta(days=1)
    start = end - timedelta(days=days)

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
        "daily": "temperature_2m_mean,relative_humidity_2m_mean",
        "timezone": "auto"
    }

    data = requests.get(url, params=params, timeout=20).json()

    df = pd.DataFrame({
        "temp": data["daily"]["temperature_2m_mean"],
        "humidity": data["daily"]["relative_humidity_2m_mean"]
    })

    return df["temp"].mean(), df["humidity"].mean()


In [26]:
def get_long_term_rainfall(lat, lon, days=180):
    end = date.today() - timedelta(days=1)
    start = end - timedelta(days=days)

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
        "daily": "precipitation_sum",
        "timezone": "auto"
    }

    data = requests.get(url, params=params, timeout=20).json()
    return sum(data["daily"]["precipitation_sum"])


In [27]:
def scientific_filter(crop, place, temp, rain, ph):
    if not fao_validate(crop, temp, rain, ph):
        return False, "Rejected by FAO ecological limits"
    if not icar_validate(crop, place):
        return False, "Rejected by ICAR regional feasibility"
    return True, "Accepted"


In [28]:
def predict_crop(place, N, P, K, ph):
    lat, lon = get_lat_lon(place)
    if lat is None:
        return " Location not found"

    temp, humidity = get_short_term_temp_humidity(lat, lon)
    rain = get_long_term_rainfall(lat, lon)

    X = [[N, P, K, temp, humidity, ph, rain]]
    X_scaled = scaler.transform(X)

    preds = model.predict_proba(X_scaled)[0]
    crop_scores = sorted(
        zip(le.classes_, preds),
        key=lambda x: x[1],
        reverse=True
    )

    for crop, score in crop_scores:
        ok, reason = scientific_filter(crop, place, temp, rain, ph)
        if ok:
            return (
                f" Temp: {temp:.2f} °C\n"
                f" Humidity: {humidity:.2f} %\n"
                f"Rainfall: {rain:.2f} mm\n\n"
                f" Recommended Crop: {crop}\n"
                f" Reason: {reason}"
            )

    return " No agronomically valid crop found"


In [30]:
interface = gr.Interface(
    fn=predict_crop,
    inputs=[
        gr.Textbox(label="Place (e.g. Surat, Gujarat)"),
        gr.Number(label="Nitrogen (N)"),
        gr.Number(label="Phosphorus (P)"),
        gr.Number(label="Potassium (K)"),
        gr.Number(label="Soil pH")
    ],
outputs=gr.Textbox(
        label="Prediction Result",
        lines=12,
        max_lines=20
    ),
    title="AI Crop Recommendation System",
    description="ML + Weather API + FAO & ICAR Scientific Validation"
)

interface.launch(share=True)


* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://50ecfcd84c51ff160a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
